# 03 — Statistical analysis (CPU only)

Reads every finished run's `preds.csv` (from notebook 02) and produces all revised tables/figures:

| Output | Paper section |
|---|---|
| `table4_mean_sd_ci.csv/.tex` — acc & macro-F1, mean ± SD over seeds, 95% hierarchical-bootstrap CI | Table 4, IV-A |
| `effects_labeled.csv/.tex` — every Δ labelled **ATTENTION**, **BALANCING**, **SE-vs-CBAM**, **ABLATION** or **INTERACTION**, with CI, bootstrap p, Holm-adjusted p, McNemar per seed | IV-B, Table 6, V-B |
| `table6_se_vs_cbam.csv` | Table 6 |
| `per_class.csv`, `tb_covid_recall.csv`, `confusion_matrices_*.png` | IV-C (move to main text) |
| `triage_operating_point.csv`, `roc_abnormal_vs_normal.png` | V-C (false-positive argument) |
| `checkpoint_best_vs_final.csv` | III-E |
| `selection_bias.csv` (rank-1 probabilities, winner's-curse optimism) | V-D |
| `ablation_loss_vs_data.csv` | III-B |
| `literature_comparison.csv` | new comparison table |
| **Revision (reviewer Q1–Q3, Q7, Q8):** family **ATTENTION-REVISION** in `effects_labeled.csv` (attention effects under lung crop, patient-grouped validation, CXR pretraining, multi-depth placement); downsample rows in the ABLATION family and `ablation_loss_vs_data.csv`; `sensitivity_variants.csv` (each variant vs its main-design reference); `checkpoint_by_validation_scheme.csv` | IV-F (new), III-B, III-E |

Headline checkpoint numbers use the original design only, so they match the submitted paper; revision runs are
reported separately.
**Statistics used.** Test-set uncertainty: stratified bootstrap over the 771 test images (class counts held fixed),
`N_BOOT` replicates, the **same** resampled indices for every model so comparisons are paired. Seed uncertainty:
seeds resampled with replacement inside each replicate (hierarchical bootstrap), seeds paired across configurations.
Two-sided bootstrap p = 2·min(P(Δ*≤0), P(Δ*≥0)). McNemar exact tests on per-image correctness, reported per seed.
Holm correction within each family of comparisons.

In [ ]:
import os, glob, json, re, itertools, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score, roc_curve

RESULT_GLOBS = ['/kaggle/input/**/results/*/metrics.json', '/kaggle/working/results/*/metrics.json']
OUT = '/kaggle/working/analysis'; os.makedirs(OUT, exist_ok=True)
N_BOOT = 2000
CKPT = 'best'             # checkpoint used for all main tables ('best' = lowest val loss, as in the paper)
ALLOW_SMOKE = False
RNG = np.random.default_rng(12345)
CLASS_NAMES = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']; K = 4
ARCH_ORDER = ['resnet50', 'densenet121', 'mobilenetv2', 'customcnn']
ARCH_LABEL = {'resnet50': 'ResNet50', 'densenet121': 'DenseNet121', 'mobilenetv2': 'MobileNetV2', 'customcnn': 'Custom CNN'}
ATT_LABEL = {'none': 'Baseline', 'se': 'SE', 'cbam': 'CBAM'}
COND_LABEL = {'imbalanced': 'Imbalanced (weighted CE)', 'imbalanced_unweighted': 'Imbalanced (plain CE)',
              'oversample': 'Oversample-only (plain CE)', 'balanced': 'Balanced (plain CE)',
              'downsample': 'Downsample-only, no augmentation (plain CE)'}
REV_KEYS = {'lungcrop', 'val', 'placement', 'pretrain'}   # revision variants (reviewer Q1–Q3, Q8); excluded from headline numbers
is_rev = lambda variant: any(f'_{k}-' in variant for k in REV_KEYS)

In [ ]:
# ── load runs ─────────────────────────────────────────────────────────────────
metas = {}
for g in RESULT_GLOBS:
    for f in glob.glob(g, recursive=True):
        m = json.load(open(f))
        if m.get('status') != 'done' or (m.get('smoke_test') and not ALLOW_SMOKE): continue
        m['dir'] = os.path.dirname(f); metas.setdefault(m['run_id'], m)
print(len(metas), 'finished runs')
RUNS, ref_paths = {}, None
for rid, m in sorted(metas.items()):
    P = pd.read_csv(os.path.join(m['dir'], 'preds.csv'))
    rec = dict(meta=m, variant=''.join(f'_{k}-{v}' for k, v in sorted(m.get('opts', {}).items())))
    for split in ['val', 'test']:
        for ck in ['best', 'final']:
            d = P[(P.split == split) & (P.ckpt == ck)].sort_values('idx')
            rec[(split, ck, 'prob')] = d[[f'p_{c}' for c in CLASS_NAMES]].values
            rec[(split, 'y')] = d.y_true.values
            if split == 'test' and ck == 'best':
                if ref_paths is None: ref_paths = d.relpath.tolist()
                assert d.relpath.tolist() == ref_paths, f'{rid}: test order differs from other runs'
    RUNS[rid] = rec
Y = RUNS[next(iter(RUNS))][('test', 'y')]; NTEST = len(Y)
cfg_of = lambda r: (r['meta']['arch'], r['meta']['attention'], r['meta']['condition'], r['variant'])
CFG = {}
for rid, r in RUNS.items(): CFG.setdefault(cfg_of(r), {})[r['meta']['seed']] = rid
print(pd.Series({k: len(v) for k, v in CFG.items()}).rename('n_seeds').to_frame().to_string())

In [ ]:
# ── metrics (point + bootstrap) ───────────────────────────────────────────────
def metrics_batch(cm):                      # cm: (..., K, K) -> dict of (...) arrays
    cm = cm.astype(float); tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp = cm.sum(-2) - tp; fn = cm.sum(-1) - tp; n = cm.sum((-1, -2))[..., None]; tn = n - tp - fp - fn
    with np.errstate(divide='ignore', invalid='ignore'):
        prec = np.nan_to_num(tp / (tp + fp)); rec = np.nan_to_num(tp / (tp + fn))
        f1 = np.nan_to_num(2 * prec * rec / (prec + rec)); spec = np.nan_to_num(tn / (tn + fp))
    out = {'acc': tp.sum(-1) / n[..., 0], 'macro_f1': f1.mean(-1), 'balanced_acc': rec.mean(-1)}
    for i, c in enumerate(CLASS_NAMES):
        out[f'prec_{c}'], out[f'rec_{c}'], out[f'f1_{c}'], out[f'spec_{c}'] = prec[..., i], rec[..., i], f1[..., i], spec[..., i]
    return out

def cm_of(y, p, idx=None):
    if idx is None: return np.bincount(y * K + p, minlength=K * K).reshape(K, K)
    code = y[idx] * K + p[idx]                                   # idx: (B, n)
    off = (np.arange(idx.shape[0]) * K * K)[:, None]
    return np.bincount((code + off).ravel(), minlength=idx.shape[0] * K * K).reshape(-1, K, K)

# stratified bootstrap indices shared by ALL runs (paired)
strata = [np.where(Y == c)[0] for c in range(K)]
BOOT = np.concatenate([s[RNG.integers(0, len(s), (N_BOOT, len(s)))] for s in strata], axis=1)

POINT, BOOTM = {}, {}
for rid, r in RUNS.items():
    for ck in ['best', 'final']:
        p = r[('test', ck, 'prob')].argmax(1)
        POINT[(rid, ck)] = metrics_batch(cm_of(Y, p))
        BOOTM[(rid, ck)] = metrics_batch(cm_of(Y, p, BOOT))           # each value: (N_BOOT,)
print('bootstrap done:', N_BOOT, 'replicates')

def seeds_of(cfg): return sorted(CFG.get(cfg, {}))

def hier(cfg, metric, ck=CKPT, seeds=None):
    """(N_BOOT,) bootstrap distribution of the seed-mean metric; seeds resampled with replacement."""
    seeds = seeds or seeds_of(cfg)
    M = np.stack([BOOTM[(CFG[cfg][s], ck)][metric] for s in seeds])          # (S, B)
    pick = RNG.integers(0, len(seeds), (N_BOOT, len(seeds)))
    return M[pick, np.arange(N_BOOT)[:, None]].mean(1)

def point_seeds(cfg, metric, ck=CKPT):
    return np.array([POINT[(CFG[cfg][s], ck)][metric] for s in seeds_of(cfg)], dtype=float)

def summarize(cfg, metric, ck=CKPT):
    v = point_seeds(cfg, metric, ck); b = hier(cfg, metric, ck)
    return dict(mean=v.mean(), sd=v.std(ddof=1) if len(v) > 1 else np.nan, n=len(v),
                lo=np.percentile(b, 2.5), hi=np.percentile(b, 97.5))

fmt = lambda s, pct=False, d=4: (f"{100*s['mean']:.2f} ± {100*s['sd']:.2f}" if pct else f"{s['mean']:.{d}f} ± {s['sd']:.{d}f}") \
      + (f" [{100*s['lo']:.2f}, {100*s['hi']:.2f}]" if pct else f" [{s['lo']:.{d}f}, {s['hi']:.{d}f}]") + f" (n={s['n']})"

## Table 4 — mean ± SD over seeds, [95% CI], (n seeds)

In [ ]:
rows = []
for a in ARCH_ORDER:
    for t in ['none', 'se', 'cbam']:
        row = {'Model': ARCH_LABEL[a], 'Attention': ATT_LABEL[t]}
        for cond in ['imbalanced', 'balanced']:
            cfg = (a, t, cond, '')
            if cfg not in CFG: continue
            for met, pct in [('acc', True), ('macro_f1', False)]:
                s = summarize(cfg, met); row[f'{cond}_{met}'] = fmt(s, pct)
                row[f'{cond}_{met}_mean'] = s['mean']
        rows.append(row)
T4 = pd.DataFrame(rows); T4.to_csv(f'{OUT}/table4_mean_sd_ci.csv', index=False)
show = [c for c in T4.columns if not c.endswith('_mean')]
display(T4[show])
with open(f'{OUT}/table4_mean_sd_ci.tex', 'w') as f: f.write(T4[show].to_latex(index=False, escape=True))

### Sensitivity: deduplicated test set (exclusion lists from notebook 01, section 3c)
`clean` removes repeat copies inside test and test images duplicated in train/val; `strict` also removes likely
same-patient images. If the results barely move, report that duplicates do not drive the conclusions.

In [ ]:
EXCL_GLOBS = ['/kaggle/input/**/audit/test_exclusion_list.csv', '/kaggle/working/audit/test_exclusion_list.csv']
ef = [f for g in EXCL_GLOBS for f in glob.glob(g, recursive=True)]
if ef:
    X = pd.read_csv(ef[0]); paths = np.array(ref_paths)
    masks_ = {'full': np.ones(NTEST, bool),
              'clean': ~np.isin(paths, X[X.list == 'clean'].relpath),
              'strict': ~np.isin(paths, X.relpath)}
    print({k: int(v.sum()) for k, v in masks_.items()}, 'test images')
    unmatched = set(X.relpath) - set(paths)
    if unmatched: print('WARNING: exclusion paths not found in preds:', list(unmatched)[:5])
    rows = []
    for cfg in sorted(CFG):
        if cfg[3] != '' or cfg[2] not in ('imbalanced', 'balanced'): continue
        r = dict(Model=ARCH_LABEL[cfg[0]], Attention=ATT_LABEL[cfg[1]], condition=cfg[2])
        for k, m in masks_.items():
            v = np.array([metrics_batch(cm_of(Y[m], RUNS[CFG[cfg][s]][('test', CKPT, 'prob')].argmax(1)[m]))['macro_f1'] for s in seeds_of(cfg)])
            r[f'macro_f1_{k}'] = f'{v.mean():.4f} ± {v.std(ddof=1) if len(v) > 1 else float("nan"):.4f}'
            r[f'_{k}'] = v.mean()
        r['max_shift'] = max(abs(r['_clean'] - r['_full']), abs(r['_strict'] - r['_full']))
        rows.append(r)
    DD = pd.DataFrame(rows); DD.to_csv(f'{OUT}/dedup_test_sensitivity.csv', index=False)
    display(DD[[c for c in DD.columns if not c.startswith('_')]])
    print(f"largest change in mean macro-F1 from deduplication: {DD.max_shift.max():.4f}")
    rk = DD.groupby('condition').apply(lambda d: (d._full.rank().values == d._clean.rank().values).mean())
    print('fraction of configurations keeping the same rank after dedup (per condition):', rk.round(2).to_dict())
else:
    print('No test_exclusion_list.csv found — attach notebook 01 output to run this check.')

## Labelled effects (ATTENTION / BALANCING / SE-vs-CBAM / ABLATION / INTERACTION)

In [ ]:
def mcnemar_p(rid_a, rid_b, ck=CKPT):
    ca = RUNS[rid_a][('test', ck, 'prob')].argmax(1) == Y; cb = RUNS[rid_b][('test', ck, 'prob')].argmax(1) == Y
    b01, b10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return (binomtest(min(b01, b10), b01 + b10, 0.5).pvalue if b01 + b10 else 1.0), b01, b10

def compare(cfg_a, cfg_b, metric='macro_f1', ck=CKPT):
    """Δ = a − b, seeds paired (common seeds only)."""
    common = sorted(set(seeds_of(cfg_a)) & set(seeds_of(cfg_b)))
    if not common: return None
    Da = np.stack([BOOTM[(CFG[cfg_a][s], ck)][metric] for s in common])
    Db = np.stack([BOOTM[(CFG[cfg_b][s], ck)][metric] for s in common])
    pick = RNG.integers(0, len(common), (N_BOOT, len(common)))
    d = (Da - Db)[pick, np.arange(N_BOOT)[:, None]].mean(1)
    pv = np.array([POINT[(CFG[cfg_a][s], ck)][metric] - POINT[(CFG[cfg_b][s], ck)][metric] for s in common])
    mc = [mcnemar_p(CFG[cfg_a][s], CFG[cfg_b][s], ck) for s in common]
    return dict(delta_mean=pv.mean(), delta_sd=pv.std(ddof=1) if len(pv) > 1 else np.nan, n_seeds=len(common),
                seeds_positive=int((pv > 0).sum()), ci_lo=np.percentile(d, 2.5), ci_hi=np.percentile(d, 97.5),
                p_boot=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())),
                mcnemar_p_per_seed=';'.join(f'{p:.3g}' for p, _, _ in mc),
                mcnemar_seeds_p_lt_05=sum(p < 0.05 for p, _, _ in mc))

def interaction(a, t, metric='macro_f1', ck=CKPT):
    """(bal − imb)_att − (bal − imb)_baseline : does balancing help the attention model less than the baseline?"""
    cf = [(a, t, 'balanced', ''), (a, t, 'imbalanced', ''), (a, 'none', 'balanced', ''), (a, 'none', 'imbalanced', '')]
    if not all(c in CFG for c in cf): return None
    common = sorted(set.intersection(*[set(seeds_of(c)) for c in cf]))
    if not common: return None
    S = [np.stack([BOOTM[(CFG[c][s], ck)][metric] for s in common]) for c in cf]
    pick = RNG.integers(0, len(common), (N_BOOT, len(common)))
    dd = ((S[0] - S[1]) - (S[2] - S[3]))[pick, np.arange(N_BOOT)[:, None]].mean(1)
    pv = np.array([(POINT[(CFG[cf[0]][s], ck)][metric] - POINT[(CFG[cf[1]][s], ck)][metric]) -
                   (POINT[(CFG[cf[2]][s], ck)][metric] - POINT[(CFG[cf[3]][s], ck)][metric]) for s in common])
    return dict(delta_mean=pv.mean(), delta_sd=pv.std(ddof=1) if len(pv) > 1 else np.nan, n_seeds=len(common),
                seeds_positive=int((pv > 0).sum()), ci_lo=np.percentile(dd, 2.5), ci_hi=np.percentile(dd, 97.5),
                p_boot=min(1.0, 2 * min((dd <= 0).mean(), (dd >= 0).mean())), mcnemar_p_per_seed='', mcnemar_seeds_p_lt_05=np.nan)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for k, i in enumerate(o): run = max(run, (m - k) * p[i]); adj[i] = min(1, run)
    return adj

E = []
def add(family, label, a, b, res):
    if res: E.append(dict(family=family, effect=label, compare=f'{a}  vs  {b}', **res))
for cond in ['imbalanced', 'balanced']:
    for a in ARCH_ORDER:
        for t in ['se', 'cbam']:
            ca, cb = (a, t, cond, ''), (a, 'none', cond, '')
            if ca in CFG and cb in CFG:
                add('ATTENTION', f'{ATT_LABEL[t]} effect | {ARCH_LABEL[a]} | {cond}', ca, cb, compare(ca, cb))
for a in ARCH_ORDER:
    for t in ['none', 'se', 'cbam']:
        ca, cb = (a, t, 'balanced', ''), (a, t, 'imbalanced', '')
        if ca in CFG and cb in CFG:
            add('BALANCING', f'Balancing effect | {ARCH_LABEL[a]} {ATT_LABEL[t]}', ca, cb, compare(ca, cb))
for cond in ['imbalanced', 'balanced']:
    for a in ARCH_ORDER:
        ca, cb = (a, 'se', cond, ''), (a, 'cbam', cond, '')
        if ca in CFG and cb in CFG:
            add('SE-vs-CBAM', f'SE − CBAM | {ARCH_LABEL[a]} | {cond}', ca, cb, compare(ca, cb))
for a in ARCH_ORDER:
    for t in ['se', 'cbam']:
        add('INTERACTION', f'Attention×Balancing | {ARCH_LABEL[a]} {ATT_LABEL[t]}  [(bal−imb)_att − (bal−imb)_base]',
            (a, t), (a, 'none'), interaction(a, t))
# attention effects inside each revision variant (lung crop, grouped validation, CXR pretraining, multi-depth placement);
# reference = same-variant baseline if it exists, else the main baseline (multi-depth has no baseline of its own)
for cfg in sorted(CFG):
    a, t, cond, v = cfg
    if v and is_rev(v) and t != 'none':
        cb = (a, 'none', cond, v) if (a, 'none', cond, v) in CFG else (a, 'none', cond, '')
        if cb in CFG: add('ATTENTION-REVISION', f'{ATT_LABEL[t]} effect | {ARCH_LABEL[a]} | {cond} | {v.strip("_")}'
                          + ('' if cb[3] else ' (vs main baseline)'), cfg, cb, compare(cfg, cb))
for a in ARCH_ORDER:
    for t in ['none', 'se', 'cbam']:
        pairs = [('Loss effect: weighted − plain CE (original data)', 'imbalanced', 'imbalanced_unweighted'),
                 ('Oversampling effect: oversample − plain CE (original data)', 'oversample', 'imbalanced_unweighted'),
                 ('Weighted CE − oversampling', 'imbalanced', 'oversample'),
                 ('Balanced set − oversample-only (downsampling Pneumonia)', 'balanced', 'oversample'),
                 ('Downsampling effect: downsample − plain CE (original data)', 'downsample', 'imbalanced_unweighted'),
                 ('Augmentation effect: balanced − downsample (augmented copies added)', 'balanced', 'downsample')]
        for lab, c1, c2 in pairs:
            ca, cb = (a, t, c1, ''), (a, t, c2, '')
            if ca in CFG and cb in CFG: add('ABLATION', f'{lab} | {ARCH_LABEL[a]} {ATT_LABEL[t]}', ca, cb, compare(ca, cb))
EFF = pd.DataFrame(E)
if len(EFF):
    EFF['p_holm'] = np.nan
    for fam, idx in EFF.groupby('family').groups.items(): EFF.loc[idx, 'p_holm'] = holm(EFF.loc[idx, 'p_boot'])
    EFF['CI_excludes_0'] = (EFF.ci_lo > 0) | (EFF.ci_hi < 0)
    EFF.to_csv(f'{OUT}/effects_labeled.csv', index=False)
    cols = ['family', 'effect', 'delta_mean', 'delta_sd', 'ci_lo', 'ci_hi', 'n_seeds', 'seeds_positive', 'p_boot', 'p_holm', 'mcnemar_seeds_p_lt_05']
    with pd.option_context('display.max_colwidth', 90, 'display.width', 250): display(EFF[cols].round(4))
    with open(f'{OUT}/effects_labeled.tex', 'w') as f: f.write(EFF[cols].round(4).to_latex(index=False, escape=True))

In [ ]:
# Table 6 replacement (SE vs CBAM, mean ± SD [CI])
rows = []
for a in ARCH_ORDER:
    row = {'Architecture': ARCH_LABEL[a]}
    for cond, tag in [('imbalanced', 'imb'), ('balanced', 'bal')]:
        for t in ['se', 'cbam']:
            if (a, t, cond, '') in CFG: row[f'{ATT_LABEL[t]} ({tag})'] = fmt(summarize((a, t, cond, ''), 'macro_f1'))
    rows.append(row)
T6 = pd.DataFrame(rows); T6.to_csv(f'{OUT}/table6_se_vs_cbam.csv', index=False); display(T6)

## Per-class results, TB/COVID recall, confusion matrices (→ main text)

In [ ]:
rows = []
for cfg in sorted(CFG):
    row = dict(arch=cfg[0], attention=cfg[1], condition=cfg[2], variant=cfg[3])
    for c in CLASS_NAMES:
        for m in ['rec', 'prec', 'f1', 'spec']:
            v = point_seeds(cfg, f'{m}_{c}'); row[f'{m}_{c}'] = f'{v.mean():.4f} ± {v.std(ddof=1) if len(v)>1 else float("nan"):.4f}'
    rows.append(row)
PC = pd.DataFrame(rows); PC.to_csv(f'{OUT}/per_class.csv', index=False)
rows = []
for cfg in sorted(CFG):
    r = dict(arch=ARCH_LABEL[cfg[0]], attention=ATT_LABEL[cfg[1]], condition=cfg[2], variant=cfg[3])
    for c in ['TURBERCULOSIS', 'COVID19', 'NORMAL']:
        s = summarize(cfg, f'rec_{c}'); r[f'recall_{c}'] = fmt(s)
    rows.append(r)
TB = pd.DataFrame(rows); TB.to_csv(f'{OUT}/tb_covid_recall.csv', index=False); display(TB)

In [ ]:
for cond in ['imbalanced', 'balanced']:
    cfgs = [(a, t, cond, '') for a in ARCH_ORDER for t in ['none', 'se', 'cbam'] if (a, t, cond, '') in CFG]
    if not cfgs: continue
    fig, ax = plt.subplots(4, 3, figsize=(12, 15))
    for k, cfg in enumerate(cfgs):
        i, j = ARCH_ORDER.index(cfg[0]), ['none', 'se', 'cbam'].index(cfg[1])
        cm = np.mean([cm_of(Y, RUNS[CFG[cfg][s]][('test', CKPT, 'prob')].argmax(1)) for s in seeds_of(cfg)], 0)
        ax[i, j].imshow(cm / cm.sum(1, keepdims=True), cmap='Blues', vmin=0, vmax=1)
        for u in range(K):
            for v in range(K):
                ax[i, j].text(v, u, f'{cm[u, v]:.1f}', ha='center', va='center', fontsize=8,
                              color='white' if cm[u, v] / cm[u].sum() > .5 else 'black')
        ax[i, j].set_xticks(range(K)); ax[i, j].set_yticks(range(K))
        ax[i, j].set_xticklabels(['COV', 'NOR', 'PNE', 'TB'], fontsize=8); ax[i, j].set_yticklabels(['COV', 'NOR', 'PNE', 'TB'], fontsize=8)
        ax[i, j].set_title(f'{ARCH_LABEL[cfg[0]]} + {ATT_LABEL[cfg[1]]} (n={len(seeds_of(cfg))})', fontsize=9)
    for a_ in ax.ravel():
        if not a_.images: a_.axis('off')
    fig.suptitle(f'Test confusion matrices, mean counts over seeds — {COND_LABEL[cond]} (rows=true, cols=pred)')
    plt.tight_layout(); plt.savefig(f'{OUT}/confusion_matrices_{cond}.png', dpi=160); plt.show()

## Operating-point analysis for the triage / false-positive argument (V-C)
Binary triage: *abnormal* = any of COVID-19, Pneumonia, TB; score = 1 − p(Normal). Thresholds marked "test-oracle"
are read off the test ROC and are descriptive only; the val-chosen threshold shows what a deployable choice would
give, but val has only 8 Normal images, so it is reported as a caveat rather than a recommendation.

In [ ]:
def triage(rid, ck=CKPT):
    pr = RUNS[rid][('test', ck, 'prob')]; y = (Y != 1).astype(int); s = 1 - pr[:, 1]; pred = pr.argmax(1)
    fpr, tpr, thr = roc_curve(y, s)
    out = dict(auc=roc_auc_score(y, s), sens_argmax=((pred != 1) & (y == 1)).sum() / y.sum(),
               spec_argmax=((pred == 1) & (y == 0)).sum() / (1 - y).sum(),
               n_normal_to_pneumonia=int(((Y == 1) & (pred == 2)).sum()), n_pneumonia_to_normal=int(((Y == 2) & (pred == 1)).sum()),
               n_abnormal_missed_as_normal=int(((Y != 1) & (pred == 1)).sum()))
    for sp in [0.90, 0.95]: out[f'sens@spec{sp:.2f}_test_oracle'] = tpr[fpr <= 1 - sp].max() if (fpr <= 1 - sp).any() else np.nan
    for se in [0.95, 0.99]: out[f'spec@sens{se:.2f}_test_oracle'] = 1 - fpr[tpr >= se].min() if (tpr >= se).any() else np.nan
    pv = RUNS[rid][('val', ck, 'prob')]; yv = (RUNS[rid][('val', 'y')] != 1).astype(int)
    if 0 < yv.sum() < len(yv):
        fv, tv, th = roc_curve(yv, 1 - pv[:, 1]); t_star = th[np.argmax(tv - fv)]
        pred_t = (s >= t_star).astype(int)
        out['val_youden_threshold'] = float(t_star)
        out['sens_at_val_threshold'] = (pred_t & y).sum() / y.sum(); out['spec_at_val_threshold'] = ((1 - pred_t) & (1 - y)).sum() / (1 - y).sum()
    for c in range(K):
        out[f'auc_ovr_{CLASS_NAMES[c]}'] = roc_auc_score((Y == c).astype(int), pr[:, c])
    return out
rows = []
for cfg in sorted(CFG):
    T = pd.DataFrame([triage(CFG[cfg][s]) for s in seeds_of(cfg)])
    r = dict(arch=ARCH_LABEL[cfg[0]], attention=ATT_LABEL[cfg[1]], condition=cfg[2], variant=cfg[3], n_seeds=len(T))
    for c in T.columns: r[c] = f'{T[c].mean():.4f} ± {T[c].std(ddof=1):.4f}' if len(T) > 1 else f'{T[c].mean():.4f}'
    rows.append(r)
TRI = pd.DataFrame(rows); TRI.to_csv(f'{OUT}/triage_operating_point.csv', index=False)
display(TRI[['arch', 'attention', 'condition', 'n_seeds', 'auc', 'sens_argmax', 'spec_argmax', 'n_normal_to_pneumonia',
             'n_pneumonia_to_normal', 'n_abnormal_missed_as_normal', 'sens@spec0.95_test_oracle']])

In [ ]:
best_cfg = max([c for c in CFG if c[2] == 'imbalanced' and c[3] == ''] or list(CFG), key=lambda c: point_seeds(c, 'macro_f1').mean())
plt.figure(figsize=(5, 5))
for s in seeds_of(best_cfg):
    pr = RUNS[CFG[best_cfg][s]][('test', CKPT, 'prob')]; f_, t_, _ = roc_curve((Y != 1).astype(int), 1 - pr[:, 1])
    plt.plot(f_, t_, alpha=.7, label=f'seed {s}')
plt.plot([0, 1], [0, 1], 'k:'); plt.xlabel('1 − specificity (Normal flagged)'); plt.ylabel('sensitivity (abnormal flagged)')
plt.title(f'Abnormal-vs-Normal ROC: {ARCH_LABEL[best_cfg[0]]}+{ATT_LABEL[best_cfg[1]]} ({best_cfg[2]})', fontsize=9)
plt.legend(); plt.tight_layout(); plt.savefig(f'{OUT}/roc_abnormal_vs_normal.png', dpi=160); plt.show()

## Best-validation vs final-epoch checkpoint (III-E)

In [ ]:
rows = []
for rid, r in RUNS.items():
    m = r['meta']
    pv_best = r[('val', 'best', 'prob')].argmax(1); yv = r[('val', 'y')]
    rows.append(dict(run_id=rid, arch=m['arch'], attention=m['attention'], condition=m['condition'], seed=m['seed'],
                     variant=r['variant'], n_val=len(yv),
                     val_scheme='patient-grouped 15% of train' if '_val-grouped' in r['variant'] else 'predefined (38 images)',
                     best_epoch=m['best_epoch'], epochs_trained=m['epochs_trained'],
                     val_acc_best=(pv_best == yv).mean(),
                     test_f1_best=POINT[(rid, 'best')]['macro_f1'], test_f1_final=POINT[(rid, 'final')]['macro_f1'],
                     test_acc_best=POINT[(rid, 'best')]['acc'], test_acc_final=POINT[(rid, 'final')]['acc']))
CK = pd.DataFrame(rows); CK['delta_f1_final_minus_best'] = CK.test_f1_final - CK.test_f1_best
CK.to_csv(f'{OUT}/checkpoint_best_vs_final.csv', index=False)
CK_all, CK = CK, CK[~CK.variant.map(is_rev)]            # headline numbers: original design only (as in the submitted paper)
print(f"runs: {len(CK)} | final > best in {(CK.delta_f1_final_minus_best > 0).mean():.0%} | mean Δ {CK.delta_f1_final_minus_best.mean():+.4f}"
      f" | mean |Δ| {CK.delta_f1_final_minus_best.abs().mean():.4f} | max |Δ| {CK.delta_f1_final_minus_best.abs().max():.4f}")
print(f"best checkpoint already at 100% val accuracy in {(CK.val_acc_best == 1).mean():.0%} of runs "
      f"(the 38-image val set cannot discriminate between them)")
display(CK.groupby(['arch', 'attention', 'condition'])[['delta_f1_final_minus_best', 'best_epoch', 'epochs_trained']]
          .agg(['mean', 'std']).round(4))
# reviewer Q8: does a larger, patient-grouped validation split make checkpoint selection informative?
if CK_all.val_scheme.nunique() > 1:
    sub = CK_all[(CK_all.arch == 'resnet50') & (CK_all.condition == 'imbalanced') & ~CK_all.variant.str.contains('lungcrop|placement|pretrain')]
    VS = sub.groupby(['val_scheme', 'attention']).agg(n_runs=('run_id', 'size'), n_val=('n_val', 'median'),
            frac_val_acc_100=('val_acc_best', lambda x: (x == 1).mean()), best_epoch=('best_epoch', 'mean'),
            test_f1_best=('test_f1_best', 'mean'), test_f1_best_sd=('test_f1_best', 'std'),
            frac_final_gt_best=('delta_f1_final_minus_best', lambda x: (x > 0).mean()),
            mean_abs_delta=('delta_f1_final_minus_best', lambda x: x.abs().mean())).reset_index()
    VS.to_csv(f'{OUT}/checkpoint_by_validation_scheme.csv', index=False); display(VS.round(4))
    from scipy.stats import spearmanr
    for sch, d in sub.groupby('val_scheme'):
        if d.val_acc_best.nunique() > 1:
            rho, pv = spearmanr(d.val_acc_best, d.test_f1_best); print(f'{sch}: Spearman(val acc, test F1) = {rho:.2f} (p={pv:.2f}, n={len(d)})')
hist_files = [(rid, os.path.join(RUNS[rid]['meta']['dir'], 'history.csv')) for rid in RUNS]
H = {rid: pd.read_csv(f) for rid, f in hist_files if os.path.exists(f)}
if any('test_macro_f1_REPORT_ONLY' in h for h in H.values()):
    sds = {rid: h.test_macro_f1_REPORT_ONLY.tail(10).std() for rid, h in H.items() if 'test_macro_f1_REPORT_ONLY' in h}
    print('SD of test macro-F1 over the last 10 epochs (median over runs):', np.median(list(sds.values())).round(4))

## Selection bias from picking the best of many runs on one test set (V-D)

In [ ]:
main = [c for c in CFG if c[2] in ('imbalanced', 'balanced') and c[3] == '']
all_seeds = sorted(set.intersection(*[set(seeds_of(c)) for c in main])) if main else []
if not all_seeds: print('No seed is shared by all main configurations yet — selection-bias analysis needs complete seeds.')
rows = []
for s in all_seeds:                                     # (a) per-seed 24-run design, test-bootstrap rank-1 probability
    F = np.stack([BOOTM[(CFG[c][s], CKPT)]['macro_f1'] for c in main])         # (C, B)
    win = np.bincount(F.argmax(0), minlength=len(main)) / N_BOOT
    obs = [POINT[(CFG[c][s], CKPT)]['macro_f1'] for c in main]; w = int(np.argmax(obs))
    others = [POINT[(CFG[main[w]][t], CKPT)]['macro_f1'] for t in all_seeds if t != s]
    rows.append(dict(seed=s, winner=f'{ARCH_LABEL[main[w][0]]}+{ATT_LABEL[main[w][1]]} ({main[w][2]})',
                     winner_test_f1=obs[w], winner_mean_f1_other_seeds=np.mean(others) if others else np.nan,
                     optimism=obs[w] - np.mean(others) if others else np.nan,
                     p_rank1_of_winner=win[w], n_configs_with_p_rank1_gt_5pct=int((win > .05).sum())))
SB = pd.DataFrame(rows); SB.to_csv(f'{OUT}/selection_bias.csv', index=False); display(SB.round(4))
if main and all_seeds:                                 # (b) rank-1 probability of the seed-mean, hierarchical
    Fm = np.stack([hier(c, 'macro_f1', seeds=all_seeds) for c in main])
    pr1 = pd.Series(np.bincount(Fm.argmax(0), minlength=len(main)) / N_BOOT,
                    index=[f'{ARCH_LABEL[c[0]]}+{ATT_LABEL[c[1]]} ({c[2]})' for c in main]).sort_values(ascending=False)
    display(pr1[pr1 > 0].rename('P(rank 1), seed-mean macro-F1').round(3).to_frame()); pr1.to_csv(f'{OUT}/rank1_probability.csv')

## Loss-vs-data ablation table (III-B)

In [ ]:
rows = []
for a in ARCH_ORDER:
    for t in ['none', 'se', 'cbam']:
        cs = [(a, t, c, '') for c in ['imbalanced_unweighted', 'imbalanced', 'oversample', 'downsample', 'balanced']]
        if sum(c in CFG for c in cs) < 3: continue
        r = {'Model': f'{ARCH_LABEL[a]} {ATT_LABEL[t]}'}
        for c in cs:
            if c in CFG: r[COND_LABEL[c[2]]] = fmt(summarize(c, 'macro_f1'))
        rows.append(r)
AB = pd.DataFrame(rows); AB.to_csv(f'{OUT}/ablation_loss_vs_data.csv', index=False); display(AB)

In [ ]:
# Sensitivity / variant runs (attention LR, unfreeze depth, MobileNetV2 baseline dropout) vs their reference config
rows = []
for cfg in CFG:
    if cfg[3] == '': continue
    ref = cfg[:3] + ('',)
    res = compare(cfg, ref) if ref in CFG else None
    if res: rows.append(dict(variant=f'{ARCH_LABEL[cfg[0]]} {ATT_LABEL[cfg[1]]} {cfg[2]} {cfg[3]}', **res))
    elif ref in CFG: print('skipped (no common seed with reference):', cfg)
if rows: display(pd.DataFrame(rows).round(4)); pd.DataFrame(rows).to_csv(f'{OUT}/sensitivity_variants.csv', index=False)

## Literature comparison (new table)
Only rows with `same_dataset_and_split=True` are directly comparable. The two rows below were checked against the
papers' abstracts/records (Nov 2026): neither uses the jtiptj Kaggle dataset or its predefined test split, so they
belong in a "context" table, not a head-to-head comparison. **Fill in / verify ref [28] from the full text before use.**

In [ ]:
LIT = pd.DataFrame([
    dict(ref='[27] Kabir et al., Heliyon 2024', classes='COVID-19/Pneumonia/TB/Normal', dataset='authors merged three public CXR datasets',
         reported='avg accuracy 0.97, precision 0.94, recall 0.97', same_dataset_and_split=False),
    dict(ref='[28] Amin et al., BSPC 2024', classes='COVID-19/TB/Pneumonia (verify)', dataset='VERIFY FROM FULL TEXT',
         reported='VERIFY FROM FULL TEXT', same_dataset_and_split=None),
    dict(ref='[29] Yen & Tsao, Sci. Rep. 2024', classes='COVID-19/Pneumonia/Normal (3-class)', dataset='COVID-19 Radiography Database (Kaggle)',
         reported='val acc 96.28%; per-class test acc 96.03/97.10/97.86%', same_dataset_and_split=False)])
if best_cfg in CFG:
    LIT = pd.concat([LIT, pd.DataFrame([dict(ref='This work', classes='4-class', dataset='jtiptj Kaggle, predefined split',
        reported=f"{ARCH_LABEL[best_cfg[0]]}+{ATT_LABEL[best_cfg[1]]}: acc {fmt(summarize(best_cfg,'acc'),True)}, "
                 f"macro-F1 {fmt(summarize(best_cfg,'macro_f1'))}", same_dataset_and_split=True)])])
LIT.to_csv(f'{OUT}/literature_comparison.csv', index=False)
with pd.option_context('display.max_colwidth', 120): display(LIT)

## Check: labelled deltas recomputed from the ORIGINAL (single-seed) Table 4
Use this for the IV-B text correction even before reruns finish.

In [ ]:
T4_paper = {('ResNet50','Baseline'):(.8967,.9521),('ResNet50','SE'):(.9567,.9544),('ResNet50','CBAM'):(.9274,.9300),
 ('DenseNet121','Baseline'):(.9285,.9451),('DenseNet121','SE'):(.9365,.9298),('DenseNet121','CBAM'):(.9207,.9356),
 ('MobileNetV2','Baseline'):(.8923,.9254),('MobileNetV2','SE'):(.8925,.9160),('MobileNetV2','CBAM'):(.8997,.9162),
 ('Custom CNN','Baseline'):(.8786,.9204),('Custom CNN','SE'):(.9131,.8990),('Custom CNN','CBAM'):(.9184,.8947)}
rows = []
for (a, t), (fi, fb) in T4_paper.items():
    bi, bb = T4_paper[(a, 'Baseline')]
    rows.append({'Model': a, 'Attention': t, 'ATTENTION effect (imb)': fi - bi, 'ATTENTION effect (bal)': fb - bb,
                 'BALANCING effect': fb - fi})
display(pd.DataFrame(rows).round(4))